In [ ]:
# =============================================================================
# BLOCKING MISS ANALYSIS — what kind of true matches are still missed after
# the original blocking (L1-L4) + Phase B (L5-L7)?
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# Decides the next fix:
#   mostly NON-LATIN names         -> a transliteration model (IndicXlit)
#   mostly NAME / ADDRESS SIMILAR  -> crowding: larger K or rare-word layer L8
#   mostly GARBLED / PARTIAL       -> rare-word layer L8 (exact distinctive words)
#
# NEW NOTEBOOK. Attach three inputs:
#   1. the competition dataset
#   2. the ckpt_train zips dataset (ckpt_train_India.zip, ckpt_train_US.zip, ...)
#   3. the Phase B TRAIN notebook's output (Add Input -> Notebooks -> that notebook)
# SETTINGS Accelerator None (CPU). Internet not required.
# RUNTIME  ~5-8 min. One cell, standalone.
# =============================================================================

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
N_EXAMPLES = 5                  # printed examples per category

import re
import time
import zipfile
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

try:
    from rapidfuzz import fuzz
    name_sim = lambda a, b: fuzz.token_set_ratio(a, b)
except ImportError:                                   # fallback, slower but fine here
    from difflib import SequenceMatcher
    name_sim = lambda a, b: 100 * SequenceMatcher(None, " ".join(sorted(set(a.split()))),
                                                  " ".join(sorted(set(b.split())))).ratio()

DATA_ROOT, INPUT_ROOT, WORK = Path(DATA_ROOT), Path(INPUT_ROOT), Path(WORK)
T0 = time.time()
log = lambda m: print(f"  [{(time.time()-T0)/60:4.1f} min] {m}", flush=True)

# ---- locate inputs (zips are unpacked once into /kaggle/working/_unzipped) ----
def find(marker):
    hits = sorted(INPUT_ROOT.rglob(marker))
    for z in sorted(INPUT_ROOT.rglob("*.zip")):
        dest = WORK / "_unzipped" / z.stem
        if not (dest / ".unpacked").exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name}")
                zf.extractall(dest)
            (dest / ".unpacked").touch()
        hits += sorted(dest.rglob(marker))
    return hits

OLD = {f.parent.name: f.parent for f in find("pairs.npz")
       if f.parent.parent.name == "ckpt_train" and (f.parent / "qids.npy").exists()}
NEW = {f.parent.name: f.parent for f in find("pairs_b.npz") if f.parent.parent.name == "ckpt_b_train"}
assert OLD, "ckpt_train not found: attach the ckpt_train zips dataset"
assert NEW, "ckpt_b_train not found: attach the Phase B TRAIN notebook's output"
COUNTRIES = sorted(set(OLD) & set(NEW))
log(f"countries: {COUNTRIES}")

# ---- raw texts + ground truth -------------------------------------------------
COLS = ["entity_id", "business_name", "business_address", "country"]
recs = pd.concat([pd.read_csv(DATA_ROOT / "train" / f"train_source{i}.tsv", sep="\t", dtype=str,
                              names=COLS, header=0, keep_default_na=False) for i in (1, 2, 3)],
                 ignore_index=True).set_index("entity_id")
gt = pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                 names=["s1", "m"], header=0, keep_default_na=False)
GT = {s: [x for x in m.split(",") if x] for s, m in zip(gt["s1"], gt["m"])}
del gt
log(f"loaded {len(recs):,} records")

LEGAL = re.compile(r"\b(llc|l l c|inc|corp|corporation|ltd|limited|pvt|private|llp|lp|pllc|co|company)\b")

def clean(s, legal=False):
    s = unicodedata.normalize("NFKD", str(s).lower()).encode("ascii", "ignore").decode()
    s = re.sub(r"\.(com|net|org|co|in)\b", " ", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    if legal:
        s = LEGAL.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def nonlatin(s):
    return any(ord(ch) > 0x024F for ch in str(s))

def addr_overlap(a, b):
    ta, tb = set(clean(a).split()), set(clean(b).split())
    return len(ta & tb) / len(ta | tb) if ta and tb else 0.0

CATS = ["not in this country's pool", "non-Latin name", "blank address",
        "name similar, address weak", "address similar, name weak",
        "name AND address similar (crowded out)", "both weak (garbled + partial)"]

def categorise(s1, cand, in_pool):
    """Primary category of a missed true pair (first matching rule wins)."""
    if not in_pool:
        return CATS[0]
    r1, r2 = recs.loc[s1], recs.loc[cand]
    if nonlatin(r2.business_name):
        return CATS[1]
    if not str(r2.business_address).strip():
        return CATS[2]
    ns = name_sim(clean(r1.business_name, True), clean(r2.business_name, True))
    ao = addr_overlap(r1.business_address, r2.business_address)
    if ns >= 80 and ao < 0.4:
        return CATS[3]
    if ao >= 0.4 and ns < 80:
        return CATS[4]
    if ns >= 80 and ao >= 0.4:
        return CATS[5]
    return CATS[6]


# ---- per country: which true pairs are still missed? --------------------------
rows, summary = [], []
for c in COUNTRIES:
    qids = np.load(OLD[c] / "qids.npy", allow_pickle=True)
    pids = np.load(OLD[c] / "pids.npy", allow_pickle=True)
    npool = len(pids)
    old = np.load(OLD[c] / "pairs.npz")
    new = np.load(NEW[c] / "pairs_b.npz")
    have = np.union1d(old["q"].astype(np.int64) * npool + old["p"],
                      new["q"].astype(np.int64) * npool + new["p"])
    ppos = pd.Series(np.arange(npool), index=pids)
    tq, tcand = [], []
    for i, s in enumerate(qids):
        for m in GT.get(s, ()):
            tq.append(i); tcand.append(m)
    tq = np.array(tq, dtype=np.int64)
    tcand = np.array(tcand, dtype=object)
    tp = ppos.reindex(tcand).to_numpy()
    in_pool = ~np.isnan(tp)
    keys = np.where(in_pool, tq * npool + np.nan_to_num(tp, nan=0).astype(np.int64), -1)
    found = in_pool & np.isin(keys, have)
    miss = np.flatnonzero(~found)
    # script split: recall for Latin vs non-Latin candidate names
    nl = np.fromiter((nonlatin(n) for n in recs["business_name"].reindex(tcand).fillna("")),
                     dtype=bool, count=len(tcand))
    summary.append((c, len(tq), int(found.sum()), int(nl.sum()), int((found & nl).sum()),
                    int((~nl).sum()), int((found & ~nl).sum())))
    t = time.time()
    for j in miss:
        rows.append((c, qids[tq[j]], tcand[j], categorise(qids[tq[j]], tcand[j], bool(in_pool[j]))))
    log(f"[{c}] {len(tq):,} true pairs, {len(miss):,} still missed ({time.time()-t:.0f}s to classify)")
    del old, new, have

ex = pd.DataFrame(rows, columns=["country", "s1", "cand", "category"])

print("\n" + "=" * 78)
print("RECALL BY SCRIPT OF THE CANDIDATE NAME  (old blocking + Phase B combined)")
print("=" * 78)
for c, n, f, nn, nf, ln, lf in summary:
    print(f"  {c:<7} overall {f/n*100:6.2f}%   non-Latin names {nf/max(nn,1)*100:6.2f}% "
          f"({nn:,} true pairs)   Latin names {lf/max(ln,1)*100:6.2f}% ({ln:,})")

print("\n" + "=" * 78)
print("STILL-MISSED TRUE PAIRS BY CATEGORY  (% of each country's misses | % of its true pairs)")
print("=" * 78)
for c, n, *_ in summary:
    sub = ex[ex.country == c]
    print(f"  {c}  ({len(sub):,} missed of {n:,} true pairs)")
    vc = sub["category"].value_counts()
    for cat in CATS:
        k = int(vc.get(cat, 0))
        print(f"    {cat:<42}{k:>9,}   {k/max(len(sub),1)*100:5.1f}% | {k/n*100:5.2f}%")

print("\n" + "=" * 78)
print("EXAMPLES")
print("=" * 78)
rng = np.random.default_rng(0)
for cat in CATS[1:]:
    sub = ex[ex.category == cat]
    if sub.empty:
        continue
    print(f"\n  --- {cat}  ({len(sub):,}) ---")
    for _, r in sub.iloc[rng.permutation(len(sub))[:N_EXAMPLES]].iterrows():
        a, b = recs.loc[r.s1], recs.loc[r.cand]
        print(f"   [{r.country}] S1  : {a.business_name}  |  {str(a.business_address)[:70]}")
        print(f"             MISS: {b.business_name}  |  {str(b.business_address)[:70]}")
out = WORK / "blocking_misses.csv"
ex.to_csv(out, index=False)
log(f"saved {out} ({len(ex):,} rows)   DONE")